# Bulk RNA-seq Explorer

Interactive companion to the pipeline's HTML report. Every cell loads a pre-computed output under `results/` and reproduces one plot or table from the report — edit cutoffs, colours, gene labels, or the plot call itself and re-run to iterate without touching Snakemake.

Run order: **Setup** first, then any section. Sections are independent after Setup.

## Setup

Loads config, contrasts, and shared helpers. `PROJECT_ROOT` defaults to the notebook's working directory (`/project` inside the container).

In [ ]:
suppressPackageStartupMessages({
  library(readr); library(dplyr); library(tidyr); library(ggplot2)
  library(ggrepel); library(yaml)
})

`%||%` <- function(a, b) if (is.null(a)) b else a

PROJECT_ROOT <- Sys.getenv("PROJECT_ROOT", normalizePath("..", mustWork = TRUE))
cfg        <- yaml::read_yaml(file.path(PROJECT_ROOT, "config", "config.yaml"))
contrasts  <- read_tsv(file.path(PROJECT_ROOT, cfg$input$contrasts_tsv %||% "config/contrasts.tsv"),
                       show_col_types = FALSE)

# Pick one contrast to explore. Swap as you like.
CONTRAST <- contrasts$contrast_id[1]
cat("Project:", basename(PROJECT_ROOT), "\nContrast:", CONTRAST, "\n")

## 1. Quality control

Source: `results/qc/qc_summary.tsv` (aggregated from MultiQC). No hard-coded pass/fail — review the metrics yourself.

In [ ]:
qc <- read_tsv(file.path(PROJECT_ROOT, "results/qc/qc_summary.tsv"), show_col_types = FALSE)
qc

## 2. Exploratory — PCA

Source: `results/exploratory/pca.rds` (precomputed prcomp on VST top-500 variable genes). Edit the colour/shape aesthetics or switch to PC3/PC4 freely.

In [ ]:
pca <- readRDS(file.path(PROJECT_ROOT, "results/exploratory/pca.rds"))
# pca$x = score matrix, pca$metadata = sample table, pca$var_explained = variance vector
scores <- as.data.frame(pca$x) |>
  tibble::rownames_to_column("sample") |>
  dplyr::left_join(pca$metadata, by = "sample")

ggplot(scores, aes(PC1, PC2, colour = condition, label = sample)) +
  geom_point(size = 3) +
  geom_text_repel(size = 3) +
  labs(x = sprintf("PC1 (%.1f%%)", 100 * pca$var_explained[1]),
       y = sprintf("PC2 (%.1f%%)", 100 * pca$var_explained[2])) +
  theme_bw()

## 3. Differential expression — volcano

Source: `results/de/<contrast>/deseq2_results.csv` (apeglm-shrunken LFC, B-H padj). Change `PADJ_CUT` / `LFC_CUT` and rerun; no DESeq2 recomputation needed.

In [ ]:
PADJ_CUT <- cfg$de$primary$padj    %||% 0.05
LFC_CUT  <- cfg$de$primary$abs_lfc %||% 1
TOP_LABEL <- 15   # how many extreme genes to annotate

de <- read_csv(file.path(PROJECT_ROOT, "results/de", CONTRAST, "deseq2_results.csv"),
               show_col_types = FALSE)

v <- de |>
  mutate(neglog10p = -log10(pmax(pvalue, 1e-300)),
         sig = case_when(
           is.na(padj)                                         ~ "ns",
           padj < PADJ_CUT & log2FoldChange >   LFC_CUT         ~ "up",
           padj < PADJ_CUT & log2FoldChange <  -LFC_CUT         ~ "down",
           TRUE                                                 ~ "ns"))
top <- v |> filter(sig != "ns") |> arrange(padj) |> head(TOP_LABEL)

ggplot(v, aes(log2FoldChange, neglog10p, colour = sig)) +
  geom_point(alpha = 0.6, size = 1) +
  geom_vline(xintercept = c(-LFC_CUT, LFC_CUT), linetype = "dashed") +
  geom_hline(yintercept = -log10(PADJ_CUT), linetype = "dashed") +
  geom_text_repel(data = top, aes(label = gene_name), size = 3, max.overlaps = 30) +
  scale_colour_manual(values = c(up = "#c0392b", down = "#2980b9", ns = "grey70")) +
  labs(title = CONTRAST, x = "log2FC (apeglm)", y = "-log10 p") +
  theme_bw()

In [ ]:
# Top DEGs by padj after the cutoff you set above.
de |>
  filter(!is.na(padj), padj < PADJ_CUT, abs(log2FoldChange) >= LFC_CUT) |>
  arrange(padj) |>
  select(gene_id, gene_name, log2FoldChange, padj, baseMean) |>
  head(30)

## 4. Enrichment — GSEA Hallmark

Source: `results/enrichment/<contrast>/gsea_hallmark.csv` (fgsea on Wald-stat-ranked genes). Swap the file name to view other MSigDB collections (`reactome`, `wikipathways`, `biocarta`, `pid`, `cgp`, `oncogenic`) or the ORA outputs (`ora_<db>.csv`).

In [ ]:
gsea <- read_csv(file.path(PROJECT_ROOT, "results/enrichment", CONTRAST, "gsea_hallmark.csv"),
                 show_col_types = FALSE)

gsea |>
  filter(padj < 0.25) |>
  slice_min(padj, n = 20) |>
  mutate(pathway = forcats::fct_reorder(pathway, NES)) |>
  ggplot(aes(NES, pathway, fill = padj)) +
  geom_col() +
  scale_fill_viridis_c(direction = -1) +
  labs(title = paste("GSEA Hallmark —", CONTRAST), y = NULL) +
  theme_bw()

## 5. TF / pathway activity

Source: `results/tf_activity/<contrast>/tf_top.tsv` (decoupler + CollecTRI, ULM scores) and `results/pathway_activity/<contrast>/` (PROGENy).

In [ ]:
tf <- read_tsv(file.path(PROJECT_ROOT, "results/tf_activity", CONTRAST, "tf_top.tsv"),
               show_col_types = FALSE)

tf |>
  slice_max(abs(score), n = 30) |>
  mutate(tf = forcats::fct_reorder(source, score)) |>
  ggplot(aes(score, tf, fill = score > 0)) +
  geom_col() +
  scale_fill_manual(values = c(`TRUE` = "#c0392b", `FALSE` = "#2980b9"), guide = "none") +
  labs(title = paste("Top TF activity —", CONTRAST), y = NULL) +
  theme_bw()

## 6. Drug repositioning

Source: `results/drug_repositioning/<contrast>/l2s2_hits.tsv` (L2S2 paired query, Ma'ayan Lab).

In [ ]:
l2s2 <- read_tsv(file.path(PROJECT_ROOT, "results/drug_repositioning", CONTRAST, "l2s2_hits.tsv"),
                 show_col_types = FALSE)
head(l2s2, 30)